## Sequencial chain

In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser

prompt1 = PromptTemplate(
    template = "write a eassy on this topic  {topic} under 500 words"
)

prompt2  = PromptTemplate(
    template = "{instructions} check and return only no. out of 10 to this eassy {essay}"
)

model = ChatGroq( model = "openai/gpt-oss-20b")

from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

class Score(BaseModel):
    score: int
    essay : str

pydanticParser = PydanticOutputParser(pydantic_object=Score)

chain =  prompt1 | model |parser | \
RunnableLambda(lambda essay: {"essay": essay, "instructions":pydanticParser.get_format_instructions()}) \
| prompt2 | model | pydanticParser



e:\Langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
res = chain.invoke({"topic":"cow"})

In [3]:
print(res.essay[:200])

The Cow: A Cornerstone of Human Civilization

The humble cow, *Bos taurus*, stands as one of humanity’s most enduring companions. From the pastoral valleys of ancient Mesopotamia to the bustling dairy


## Parallel Chain

In [4]:
from langchain_core.runnables import RunnableParallel 

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

prompt = PromptTemplate(
    template = "give me one fun facts about  {topic}"
    )
prompt2  = PromptTemplate(
    template = "{instructions} summarize the following facts into one sentence: {first} \n {second}"
)

model = ChatGroq( model = "openai/gpt-oss-20b")
parser = StrOutputParser()

parallel_chain = RunnableParallel(
    {"first": prompt | model | parser,
     "second": prompt | model | parser}
)

class Summary(BaseModel):
    summary: str
    first: str
    second: str

pydanticParser = PydanticOutputParser(pydantic_object=Summary)
# merged = (
#     parallel_chain
#     | RunnableLambda(lambda x: {
#         **x,
#         "summary": (prompt2 | model | pydanticParser).invoke({"first": x["first"],
#                          "second": x["second"],"instructions": pydanticParser.get_format_instructions()})
#         })
#     )

merged = (
    parallel_chain|RunnableLambda(lambda x: {
        "first": x["first"],
        "second": x["second"],
        "instructions": pydanticParser.get_format_instructions(),
    }) \
    |prompt2 | model | pydanticParser
    )




In [5]:
res = merged.invoke({"topic":"elephant"})

In [6]:
res

Summary(summary='An adult elephant’s trunk contains about 40,000 muscles—roughly the same as a human’s entire body—enabling it to perform delicate tasks like picking up a single grain of rice or powerful ones such as uprooting a tree, while elephants can sense earthquakes up to 70 miles away via seismic vibrations in their feet and trunks, often changing behavior before the quake.', first='An adult elephant’s trunk contains about 40,000 muscles—roughly the same as a human’s entire body—enabling it to perform delicate tasks like picking up a single grain of rice or powerful ones such as uprooting a tree.', second='Elephants can sense earthquakes up to 70 miles away via seismic vibrations in their feet and trunks, often changing behavior before the quake.')

## Conditional Chain


In [7]:
from langchain_core.runnables import RunnableBranch,RunnableLambda,RunnablePassthrough
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

class SentimentResponse(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="The sentiment of the text")
    
prompt = PromptTemplate(
    template = "{instruction} give the sentiment of the following text: {text} "
)
model = ChatGroq( model = "openai/gpt-oss-20b")
parser = PydanticOutputParser(pydantic_object=SentimentResponse)



chain1 = prompt | model | parser | RunnableLambda(lambda x: {"sentiment": x.sentiment})
conditional_chain = RunnableBranch(
    (lambda x: x["sentiment"] == "positive",
        RunnableLambda(lambda x: f"The text is {x['sentiment']}. Great job!")),
    (lambda x: x["sentiment"] == "negative",
        RunnableLambda(lambda x: f"The text is {x['sentiment']}. Try to be more positive!")),
    (lambda x: x["sentiment"] == "neutral",
        RunnableLambda(lambda x: f"The text is {x['sentiment']}. It's balanced.")),
    RunnableLambda(lambda x: "Sentiment could not be determined.")
)

chain = chain1 | conditional_chain


In [8]:
res = chain.invoke({"instruction":parser.get_format_instructions(), "text":"indias capital is delhi!"})
res

"The text is neutral. It's balanced."

In [9]:
!pip install grandalf


[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
      +----------+       
      | ChatGroq |       
      +----------+       
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Lambda |        
       +--------+        
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +-------